[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_cleaning.ipynb)

# DataSage Stage 1: Cleaning GRPO Training

Pre-builds dataset with real environment observations (no `rollout_func`).
Reward functions replay the env with stored seeds for context-matched evaluation.

**Fully self-contained** — runs in Colab with no repo clone or project imports.

**Requirements:** GPU (A100/H100), `HF_TOKEN`, `WANDB_API_KEY` env vars.

In [ ]:
# Setup: install dependencies
!pip install -q unsloth trl datasets wandb requests

In [ ]:
# Load API keys from Colab Secrets or set manually
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

In [ ]:
# Config + parser + reward helpers (all inlined, no project imports)
import json
import re
import requests

# ---- Config constants ----
WANDB_PROJECT = "datasage"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
ENV_URL = "https://ricalanis-datasage-cleaning.hf.space"
HF_REPO = "ricalanis/datasage-qwen-cleaning"

LEARNING_RATE = 5e-6
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
NUM_GENERATIONS = 4
MAX_COMPLETION_LENGTH = 256
MAX_PROMPT_LENGTH = 512


# ---- Completion text extraction ----
def _get_text(completion) -> str:
    """Extract text from a completion (str or chat message list)."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        # Chat format: [{"role": "assistant", "content": "..."}]
        return completion[-1]["content"] if completion else ""
    return str(completion)


# ---- Parser ----
def _extract_column(text: str) -> str:
    """Try to extract a column name from text."""
    quoted = re.findall(r'["\']([\w]+)["\']', text)
    if quoted:
        return quoted[0]
    camel = re.findall(r'\b([A-Z][a-z]+(?:[A-Z][a-z]+)+)\b', text)
    if camel:
        return camel[0]
    return ""


def parse_cleaning_action(text: str) -> dict:
    """Parse model output into a cleaning action dict."""
    json_match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "operation" in data and "column" in data:
                return data
        except json.JSONDecodeError:
            pass

    text_lower = text.lower()
    if "fill" in text_lower or "null" in text_lower:
        col = _extract_column(text)
        return {"operation": "fill_null", "column": col, "value": "median", "params": {}}
    elif "type" in text_lower or "cast" in text_lower or "convert" in text_lower:
        col = _extract_column(text)
        return {"operation": "fix_type", "column": col, "value": "numeric", "params": {}}
    elif "duplicate" in text_lower or "dedup" in text_lower:
        return {"operation": "remove_duplicate", "column": "", "params": {}}
    elif "standard" in text_lower or "normalize" in text_lower:
        col = _extract_column(text)
        return {"operation": "standardize", "column": col, "params": {}}
    elif "trim" in text_lower or "whitespace" in text_lower:
        col = _extract_column(text)
        return {"operation": "trim", "column": col, "params": {}}
    elif "typo" in text_lower or "correct" in text_lower:
        col = _extract_column(text)
        return {"operation": "correct_typo", "column": col, "params": {}}

    return {"operation": "fill_null", "column": "", "value": "median", "params": {}}


# ---- Reward helpers ----
_ACTION_JSON_RE = re.compile(r'\{[^{}]*"operation"[^{}]*\}')
CLEANING_VALID_OPS = {"fill_null", "fix_type", "remove_duplicate",
                      "standardize", "trim", "correct_typo"}


def cleaning_json_format_reward(completions, **kwargs) -> list[float]:
    """Reward well-formed JSON cleaning actions."""
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        match = _ACTION_JSON_RE.search(text)
        if match:
            try:
                data = json.loads(match.group())
                if data.get("operation") in CLEANING_VALID_OPS and "column" in data:
                    rewards.append(1.0)
                elif data.get("operation") in CLEANING_VALID_OPS:
                    rewards.append(0.6)
                else:
                    rewards.append(0.3)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


print(f"Environment URL: {ENV_URL}")
print(f"Base model: {BASE_MODEL}")

In [ ]:
# Model loading via Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {BASE_MODEL}")

In [ ]:
# System prompt and task descriptions
SYSTEM_PROMPT = """\
You are a data quality agent. You clean enterprise datasets across multiple \
domains (HR, Sales, Project Management, IT Operations).

Analyze the data quality report and data preview below, then respond with a \
JSON cleaning action:
{"operation": "<op>", "column": "<col>", "value": "<val>", "params": {}}

Available operations:
- fill_null: Fill null values (value can be "median", "mode", or a specific value)
- fix_type: Fix type mismatches in a column (cast to proper type)
- remove_duplicate: Remove duplicate rows
- standardize: Standardize column values (strip whitespace, normalize case)
- trim: Trim whitespace from column values
- correct_typo: Correct typos in categorical values

Identify the most impactful issue and act."""

TASK_DESCRIPTIONS = [
    "Clean the data to maximize the data quality score.",
    "Fix the most impactful data quality issue in this dataset.",
    "Analyze the DQ report and apply the best cleaning operation.",
    "This data has quality issues. Identify and fix the worst one.",
    "Improve the data quality score as much as possible with one operation.",
    "Look at the null counts, type issues, and duplicates. Fix the biggest problem.",
    "The data quality score is low. Apply the cleaning operation with the highest impact.",
    "Examine the data preview and quality report. What single operation improves quality most?",
]

In [ ]:
# Dataset: pre-built with real environment observations
import random
from datasets import Dataset

random.seed(42)
SEEDS = [42 + i for i in range(64)]
prompts = []

print("Building dataset with real environment observations...")
for i, seed in enumerate(SEEDS):
    task_desc = random.choice(TASK_DESCRIPTIONS)
    resp = requests.post(f"{ENV_URL}/reset", json={"seed": seed})
    resp.raise_for_status()
    obs = resp.json()["observation"]

    prompts.append([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Domain: {obs['domain']}\n\n"
            f"Data Quality Report:\n{obs['dq_report']}\n"
            f"DQ Score: {obs['dq_score']:.4f}\n\n"
            f"Columns:\n{obs['columns_info']}\n\n"
            f"Data Preview:\n{obs['data_preview']}\n\n"
            f"Task: {task_desc}"
        )},
    ])
    if (i + 1) % 16 == 0:
        print(f"  Built {i + 1}/64 examples")

dataset = Dataset.from_dict({
    "prompt": prompts,
    "seed": SEEDS,
})
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Environment reward function (calls env directly with stored seeds)
def cleaning_env_reward(completions, **kwargs) -> list[float]:
    """Primary reward: replay env with stored seed, step with parsed action."""
    seeds = kwargs.get("seed", [0] * len(completions))
    rewards = []
    for completion, seed in zip(completions, seeds):
        try:
            text = _get_text(completion)
            action_dict = parse_cleaning_action(text)
            requests.post(f"{ENV_URL}/reset", json={"seed": int(seed)})
            resp = requests.post(f"{ENV_URL}/step", json={"action": action_dict})
            resp.raise_for_status()
            rewards.append(float(resp.json().get("reward", 0.0)))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

In [ ]:
# GRPO training config
from trl import GRPOConfig

training_args = GRPOConfig(
    output_dir="./outputs/cleaning-grpo",
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    logging_steps=1, save_steps=50, bf16=True,
    report_to="wandb", run_name="datasage-cleaning-grpo-v2",
)
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

In [ ]:
# Create trainer and train
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args,
    train_dataset=dataset,
    reward_funcs=[cleaning_env_reward, cleaning_json_format_reward],
)
print("Starting Stage 1 (Cleaning) GRPO training...")
trainer.train()

In [ ]:
# Save and push to Hub
output_dir = "./outputs/cleaning-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

print(f"Pushing to Hub: {HF_REPO}")
trainer.push_to_hub(HF_REPO)
print(f"Model pushed to https://huggingface.co/{HF_REPO}")